# 05 — Exploratory Data Analysis

**Objective:** Generate comprehensive visualizations and business insights across customer demographics, loyalty behavior, flight activity, and churn patterns.

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.config import *

pd.set_option('display.max_columns', None)

# Color palette
COLORS = px.colors.qualitative.Set2
DARK_TEMPLATE = 'plotly_dark'

In [2]:
# Load data
loyalty = pd.read_csv(CLEANED_DATA_DIR / 'loyalty_with_churn.csv')
flights = pd.read_csv(CLEANED_DATA_DIR / 'flights_cleaned.csv')
features = pd.read_csv(FEATURES_DIR / 'customer_features.csv')

flights['YearMonth'] = pd.to_datetime(flights['YearMonth'])
loyalty['Enrollment_Date'] = pd.to_datetime(loyalty['Enrollment_Date'])

print(f'Loyalty: {loyalty.shape}  |  Flights: {flights.shape}  |  Features: {features.shape}')

Loyalty: (16737, 25)  |  Flights: (389065, 9)  |  Features: (16737, 45)


---
## 1. Customer Demographics Insights

### 1.1 Gender Distribution

In [4]:
gender_counts = loyalty['Gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']

fig = px.pie(gender_counts, values='Count', names='Gender', title='Customer Gender Distribution',
             color_discrete_sequence=COLORS, hole=0.4)
fig.update_layout(template=DARK_TEMPLATE, height=400)
fig.show()

### 1.2 Education Distribution

In [6]:
edu_counts = loyalty['Education'].value_counts().reindex(EDUCATION_ORDER).reset_index()
edu_counts.columns = ['Education', 'Count']

fig = px.bar(edu_counts, x='Education', y='Count', title='Education Level Distribution',
             color='Education', color_discrete_sequence=COLORS,
             text='Count')
fig.update_traces(textposition='outside')
fig.update_layout(template=DARK_TEMPLATE, height=420, showlegend=False)
fig.show()

### 1.3 Salary Distribution

In [7]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Salary Histogram', 'Salary by Education'])

fig.add_trace(go.Histogram(x=loyalty['Salary'], nbinsx=50, marker_color='#636EFA', name='Salary'), row=1, col=1)

for i, edu in enumerate(EDUCATION_ORDER):
    data = loyalty[loyalty['Education'] == edu]['Salary']
    fig.add_trace(go.Box(y=data, name=edu, marker_color=COLORS[i % len(COLORS)]), row=1, col=2)

fig.update_layout(template=DARK_TEMPLATE, height=400, title='Salary Analysis', showlegend=False)
fig.show()

### 1.4 Marital Status

In [8]:
marital_counts = loyalty['Marital Status'].value_counts().reset_index()
marital_counts.columns = ['Status', 'Count']

fig = px.bar(marital_counts, x='Status', y='Count', title='Marital Status Distribution',
             color='Status', color_discrete_sequence=COLORS, text='Count')
fig.update_traces(textposition='outside')
fig.update_layout(template=DARK_TEMPLATE, height=400, showlegend=False)
fig.show()

### 1.5 Geographic Distribution

In [9]:
province_counts = loyalty['Province'].value_counts().head(10).reset_index()
province_counts.columns = ['Province', 'Count']

fig = px.bar(province_counts, x='Count', y='Province', orientation='h',
             title='Top 10 Provinces by Customer Count',
             color='Count', color_continuous_scale='Blues', text='Count')
fig.update_traces(textposition='outside')
fig.update_layout(template=DARK_TEMPLATE, height=450, yaxis={'categoryorder': 'total ascending'})
fig.show()

---
## 2. Loyalty Program Insights

### 2.1 Loyalty Tier Performance

In [11]:
tier_stats = loyalty.groupby('Loyalty Card').agg(
    Customer_Count=(PK, 'count'),
    Avg_CLV=('CLV', 'mean'),
    Avg_Salary=('Salary', 'mean'),
    Churn_Rate=('Churn', 'mean'),
    Avg_Tenure=('Tenure_Months', 'mean'),
).reset_index()
tier_stats['Churn_Rate'] = (tier_stats['Churn_Rate'] * 100).round(1)

display(tier_stats)

fig = make_subplots(rows=1, cols=3, subplot_titles=['Customer Count', 'Average CLV', 'Churn Rate %'])

for i, tier in enumerate(LOYALTY_CARD_ORDER):
    row = tier_stats[tier_stats['Loyalty Card'] == tier]
    fig.add_trace(go.Bar(x=[tier], y=row['Customer_Count'], name=tier, marker_color=COLORS[i], showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=[tier], y=row['Avg_CLV'], name=tier, marker_color=COLORS[i], showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(x=[tier], y=row['Churn_Rate'], name=tier, marker_color=COLORS[i], showlegend=False), row=1, col=3)

fig.update_layout(template=DARK_TEMPLATE, height=400, title='Loyalty Tier Performance Comparison')
fig.show()

,Loyalty Card,Customer_Count,Avg_CLV,Avg_Salary,Churn_Rate,Avg_Tenure
0,Aurora,3429,10506.129102,77558.697579,17.2,35.439778
1,Nova,5671,7949.977194,77390.038794,17.1,35.241227
2,Star,7637,6683.219659,77906.031033,16.2,35.613854


### 2.2 CLV Distribution

In [12]:
fig = px.violin(loyalty, x='Loyalty Card', y='CLV', color='Loyalty Card',
                title='CLV Distribution by Loyalty Tier',
                color_discrete_sequence=COLORS, box=True)
fig.update_layout(template=DARK_TEMPLATE, height=450)
fig.show()

### 2.3 Enrollment Trends

In [13]:
enrollment_trend = loyalty.groupby(loyalty['Enrollment_Date'].dt.to_period('Q').astype(str))[PK].count().reset_index()
enrollment_trend.columns = ['Quarter', 'Enrollments']

fig = px.line(enrollment_trend, x='Quarter', y='Enrollments', title='Quarterly Enrollment Trend',
              markers=True)
fig.update_layout(template=DARK_TEMPLATE, height=400)
fig.update_xaxes(tickangle=45)
fig.show()

---
## 3. Behavioral Insights

### 3.1 Flight Activity Trends Over Time

In [14]:
monthly_activity = flights.groupby(flights['YearMonth'].dt.to_period('M').astype(str)).agg(
    Total_Flights=('Total Flights', 'sum'),
    Total_Distance=('Distance', 'sum'),
    Active_Customers=(PK, 'nunique'),
    Total_Points_Earned=('Points Accumulated', 'sum'),
).reset_index()
monthly_activity.columns = ['Month', 'Total_Flights', 'Total_Distance', 'Active_Customers', 'Points_Earned']

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Total Flights per Month', 'Total Distance per Month',
    'Active Customers per Month', 'Points Earned per Month'
])

fig.add_trace(go.Scatter(x=monthly_activity['Month'], y=monthly_activity['Total_Flights'], 
                         mode='lines', line=dict(color='#636EFA'), name='Flights'), row=1, col=1)
fig.add_trace(go.Scatter(x=monthly_activity['Month'], y=monthly_activity['Total_Distance'], 
                         mode='lines', line=dict(color='#EF553B'), name='Distance'), row=1, col=2)
fig.add_trace(go.Scatter(x=monthly_activity['Month'], y=monthly_activity['Active_Customers'], 
                         mode='lines', line=dict(color='#00CC96'), name='Customers'), row=2, col=1)
fig.add_trace(go.Scatter(x=monthly_activity['Month'], y=monthly_activity['Points_Earned'], 
                         mode='lines', line=dict(color='#AB63FA'), name='Points'), row=2, col=2)

fig.update_layout(template=DARK_TEMPLATE, height=600, title='Flight Activity Trends', showlegend=False)
fig.update_xaxes(tickangle=45)
fig.show()

### 3.2 Redemption Patterns

In [15]:
# Monthly redemption heatmap by year
flights['Year_val'] = flights['YearMonth'].dt.year
flights['Month_val'] = flights['YearMonth'].dt.month
redemption_heatmap = flights.groupby(['Year_val', 'Month_val'])['Points Redeemed'].sum().reset_index()
redemption_pivot = redemption_heatmap.pivot(index='Year_val', columns='Month_val', values='Points Redeemed').fillna(0)

fig = px.imshow(redemption_pivot, 
                labels=dict(x='Month', y='Year', color='Points Redeemed'),
                title='Points Redemption Heatmap (Year × Month)',
                color_continuous_scale='YlOrRd',
                aspect='auto')
fig.update_layout(template=DARK_TEMPLATE, height=400)
fig.show()

flights.drop(columns=['Year_val', 'Month_val'], inplace=True)

### 3.3 Flights vs Points Relationship

In [16]:
# Customer-level aggregation
customer_agg = flights.groupby(PK).agg(
    Total_Flights=('Total Flights', 'sum'),
    Total_Points=('Points Accumulated', 'sum'),
    Total_Distance=('Distance', 'sum'),
).reset_index()
customer_agg = customer_agg.merge(loyalty[[PK, 'Loyalty Card', 'Churn']], on=PK, how='left')

fig = px.scatter(customer_agg.sample(min(3000, len(customer_agg)), random_state=42), 
                 x='Total_Flights', y='Total_Points', color='Loyalty Card',
                 title='Total Flights vs Points Accumulated (by Tier)',
                 color_discrete_sequence=COLORS, opacity=0.6,
                 hover_data=['Total_Distance'])
fig.update_layout(template=DARK_TEMPLATE, height=500)
fig.show()

---
## 4. Churn Insights

### 4.1 Overall Churn Rate

In [17]:
churn_counts = loyalty['Churn'].value_counts().reset_index()
churn_counts.columns = ['Status', 'Count']
churn_counts['Status'] = churn_counts['Status'].map({0: 'Active', 1: 'Churned'})

fig = px.pie(churn_counts, values='Count', names='Status', title='Overall Churn Distribution',
             color_discrete_sequence=['#00CC96', '#EF553B'], hole=0.5)
fig.update_layout(template=DARK_TEMPLATE, height=400)
fig.show()

print(f'Overall churn rate: {loyalty["Churn"].mean()*100:.1f}%')
print(f'Churned customers: {loyalty["Churn"].sum():,}')
print(f'Active customers: {(loyalty["Churn"]==0).sum():,}')

Overall churn rate: 16.7%
Churned customers: 2,794
Active customers: 13,943


### 4.2 Churn by Loyalty Tier

In [19]:
churn_tier = loyalty.groupby(['Loyalty Card', 'Churn']).size().reset_index(name='Count')
churn_tier['Churn_Label'] = churn_tier['Churn'].map({0: 'Active', 1: 'Churned'})

fig = px.bar(churn_tier, x='Loyalty Card', y='Count', color='Churn_Label',
             title='Churn Distribution by Loyalty Tier', barmode='group',
             color_discrete_sequence=['#00CC96', '#EF553B'],
             text='Count')
fig.update_traces(textposition='outside')
fig.update_layout(template=DARK_TEMPLATE, height=400)
fig.show()

### 4.3 Churn by Demographics

In [20]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['By Gender', 'By Education', 'By Marital Status'])

# By Gender
churn_gender = loyalty.groupby('Gender')['Churn'].mean().reset_index()
churn_gender['Churn'] = churn_gender['Churn'] * 100
fig.add_trace(go.Bar(x=churn_gender['Gender'], y=churn_gender['Churn'], 
                     marker_color=['#636EFA', '#EF553B'], showlegend=False), row=1, col=1)

# By Education
churn_edu = loyalty.groupby('Education')['Churn'].mean().reindex(EDUCATION_ORDER).reset_index()
churn_edu['Churn'] = churn_edu['Churn'] * 100
fig.add_trace(go.Bar(x=churn_edu['Education'], y=churn_edu['Churn'], 
                     marker_color=COLORS[:len(churn_edu)], showlegend=False), row=1, col=2)

# By Marital Status
churn_marital = loyalty.groupby('Marital Status')['Churn'].mean().reset_index()
churn_marital['Churn'] = churn_marital['Churn'] * 100
fig.add_trace(go.Bar(x=churn_marital['Marital Status'], y=churn_marital['Churn'], 
                     marker_color=COLORS[:len(churn_marital)], showlegend=False), row=1, col=3)

fig.update_yaxes(title_text='Churn Rate %')
fig.update_layout(template=DARK_TEMPLATE, height=400, title='Churn Rate by Demographics')
fig.show()

### 4.4 Churn by Activity Level

In [21]:
# Create activity quartiles
features_with_churn = features.copy()
features_with_churn['Activity_Quartile'] = pd.qcut(
    features_with_churn['Flights_12M'].clip(lower=0), q=4, labels=['Low', 'Medium', 'High', 'Very High'],
    duplicates='drop'
)

churn_activity = features_with_churn.groupby('Activity_Quartile')['Churn'].agg(['mean', 'count']).reset_index()
churn_activity['Churn Rate %'] = (churn_activity['mean'] * 100).round(1)

fig = px.bar(churn_activity, x='Activity_Quartile', y='Churn Rate %',
             title='Churn Rate by Flight Activity Level (12M)',
             color='Churn Rate %', color_continuous_scale='RdYlGn_r',
             text='Churn Rate %')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template=DARK_TEMPLATE, height=400)
fig.show()

### 4.5 Churn by CLV Segments

In [22]:
features_with_churn['CLV_Segment'] = pd.qcut(
    features_with_churn['CLV'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'],
    duplicates='drop'
)

churn_clv = features_with_churn.groupby('CLV_Segment')['Churn'].agg(['mean', 'count']).reset_index()
churn_clv['Churn Rate %'] = (churn_clv['mean'] * 100).round(1)

fig = px.bar(churn_clv, x='CLV_Segment', y='Churn Rate %',
             title='Churn Rate by CLV Segment',
             color='Churn Rate %', color_continuous_scale='RdYlGn_r',
             text='Churn Rate %')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template=DARK_TEMPLATE, height=400)
fig.show()

### 4.6 Feature Correlations

In [23]:
# Top correlations with churn
feature_cols = [c for c in features.columns if c not in [PK, 'Churn']]
correlations = features[feature_cols + ['Churn']].corr()['Churn'].drop('Churn').sort_values()

top_bottom = pd.concat([correlations.head(10), correlations.tail(10)])

fig = px.bar(x=top_bottom.values, y=top_bottom.index, orientation='h',
             title='Top 20 Feature Correlations with Churn',
             color=top_bottom.values, color_continuous_scale='RdBu_r',
             labels={'x': 'Correlation', 'y': 'Feature'})
fig.update_layout(template=DARK_TEMPLATE, height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()